# RAG from scratch

Pipeline: **load** → chunk → embed → store → retrieve → generate

## Step 1: Load
Goal: turn each PDF into plain text, and keep track of **where the text came from** (file + page).
That "where" is called **metadata**. We need it later to cite sources and to check retrieval against `eval/questions.json`.

In [ ]:
from pathlib import Path
from pypdf import PdfReader

DATA_DIR = Path("../data/raw")  # notebooks run fraom the notebooks/ folder

def load_pdfs(folder):
    """Return one record per PDF page: {text, source, page}."""
    pages = []
    for pdf_path in sorted(folder.glob("*.pdf")):
        reader = PdfReader(pdf_path)
        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""  # scanned/image pages return None
            pages.append({"text": text, "source": pdf_path.name, "page": page_num})
    return pages

pages = load_pdfs(DATA_DIR)
print(f"Loaded {len(pages)} pages")
for p in pages:
    print(f"{p['source']:40} page {p['page']}  {len(p['text']):5} chars")

Loaded 9 pages
01_caching_and_cdn.pdf                   page 1   2475 chars
01_caching_and_cdn.pdf                   page 2   2956 chars
01_caching_and_cdn.pdf                   page 3   1811 chars
02_project_falcon_databases.pdf          page 1   2578 chars
02_project_falcon_databases.pdf          page 2   2761 chars
02_project_falcon_databases.pdf          page 3   1322 chars
03_shopkart_checkout_case_study.pdf      page 1   2704 chars
03_shopkart_checkout_case_study.pdf      page 2   2308 chars
03_shopkart_checkout_case_study.pdf      page 3    306 chars


### Inspect the raw text
Always look at what the loader produced. Check: did the **table** survive? Are there broken words, extra line breaks, or headers/footers repeated on every page?

In [3]:
print(pages[0]["text"][:3000])

Caching and CDN Design Guide
Halcyon Labs Platform Engineering · Internal standard PE-CACHE-07 · Revision 3
1. Why we cache
Caching is the practice of storing the result of an expensive operation so that later requests can be
served faster and more cheaply. At Halcyon Labs almost every read path passes through at least
one cache layer before it touches a database. A well-designed cache reduces latency, protects
downstream systems from load spikes and lowers infrastructure cost. A badly designed cache
serves stale data, hides bugs and fails in surprising ways during incidents.
This guide explains the general principles first and then describes the specific defaults that Halcyon
Labs teams must follow. Where this document says must, a deviation needs approval from the
Platform Architecture Review, which meets every second Thursday.
2. Cache placement and tiers
Caches can live in several places: in the client (browser or mobile app), at the edge (a CDN), in a
shared distributed cache such